# SupportIQ — Stage 1.2: Essential Data Profiling & Sequence Lengths

> **Goal:** Verify class distributions, sequence lengths with Qwen tokenizer, and template near-duplicates to guide fine-tuning hyperparameters.


### 1. Setup & Load Data

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import polars as pl

from supportiq.data.load import load_raw_dataframe

df = load_raw_dataframe()
print(f"Loaded {df.height:,} rows for profiling.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded 26,872 rows for profiling.


### 2. Category & Intent Distributions
Verify class balance across the 11 categories and 27 intents.

In [2]:
cat_dist = (
    df.group_by("category")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("percentage"))
    .sort("count", descending=True)
)
print("Category Distribution:")
print(cat_dist)
print(
    f"\nUnique categories: {df['category'].n_unique()} | Unique intents: {df['intent'].n_unique()}"
)

Category Distribution:
shape: (11, 3)
┌──────────────┬───────┬────────────┐
│ category     ┆ count ┆ percentage │
│ ---          ┆ ---   ┆ ---        │
│ str          ┆ u32   ┆ f64        │
╞══════════════╪═══════╪════════════╡
│ ACCOUNT      ┆ 5986  ┆ 22.28      │
│ ORDER        ┆ 3988  ┆ 14.84      │
│ REFUND       ┆ 2992  ┆ 11.13      │
│ INVOICE      ┆ 1999  ┆ 7.44       │
│ CONTACT      ┆ 1999  ┆ 7.44       │
│ …            ┆ …     ┆ …          │
│ FEEDBACK     ┆ 1997  ┆ 7.43       │
│ DELIVERY     ┆ 1994  ┆ 7.42       │
│ SHIPPING     ┆ 1970  ┆ 7.33       │
│ SUBSCRIPTION ┆ 999   ┆ 3.72       │
│ CANCEL       ┆ 950   ┆ 3.54       │
└──────────────┴───────┴────────────┘

Unique categories: 11 | Unique intents: 27


### 3. Placeholder Slots Detection
Confirm intentional template slots (like `{{Order Number}}`) without dumping thousands of lines.

In [3]:
import re

placeholder_regex = re.compile(r"\{\{([^}]+)\}\}")
all_placeholders = set()
for text in df["instruction"].to_list() + df["response"].to_list():
    all_placeholders.update(placeholder_regex.findall(text))

print(f"Total unique template slot types: {len(all_placeholders)}")
print("Sample slot entities:", [f"{{{{{p}}}}}" for p in sorted(all_placeholders)[:5]])

Total unique template slot types: 391
Sample slot entities: ['{{Access Key}}', '{{Access Key Recovery}}', '{{Access Key Reset Page URL}}', '{{Access Key Retrieval}}', '{{Account}}']


### 4. Sequence Length Profiling (Qwen Tokenizer)
Measure token lengths to determine `max_seq_length` for SFT fine-tuning.

In [4]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

sample_inst = df["instruction"].to_list()
sample_resp = df["response"].to_list()

inst_lens = [len(t) for t in tokenizer(sample_inst, add_special_tokens=False)["input_ids"]]
resp_lens = [len(t) for t in tokenizer(sample_resp, add_special_tokens=False)["input_ids"]]
total_lens = [i + r for i, r in zip(inst_lens, resp_lens, strict=True)]

print("Qwen Token Length Percentiles (Total Sequence = User + Assistant):")
print(f"  - Min:    {np.min(total_lens)} tokens")
print(f"  - Median: {np.median(total_lens):.0f} tokens")
print(f"  - P95:    {np.percentile(total_lens, 95):.0f} tokens")
print(f"  - Max:    {np.max(total_lens)} tokens")
print(
    f"  - Truncation rate at max_seq_length=512: {sum(1 for t in total_lens if t > 512) / len(total_lens) * 100:.2f}%"
)

Qwen Token Length Percentiles (Total Sequence = User + Assistant):
  - Min:    18 tokens
  - Median: 115 tokens
  - P95:    267 tokens
  - Max:    490 tokens
  - Truncation rate at max_seq_length=512: 0.00%


### 5. Template Near-Duplicate Check
Demonstrate how paraphrases cluster around the same template.

In [5]:
from datasketch import MinHash, MinHashLSH


def shingle_text(s: str) -> set[str]:
    w = s.lower().split()
    return {" ".join(w[i : i + 3]) for i in range(len(w) - 2)} if len(w) >= 3 else {s.lower()}


lsh = MinHashLSH(threshold=0.80, num_perm=128)
sample_subset = df["instruction"][:1000].to_list()
minhashes = []

for idx, text in enumerate(sample_subset):
    m = MinHash(num_perm=128)
    for shingle in shingle_text(text):
        m.update(shingle.encode("utf8"))
    minhashes.append(m)
    lsh.insert(f"row_{idx}", m)

# Query one example template
matches = lsh.query(minhashes[0])
print(f"Seed Question: {sample_subset[0]}")
print(f"Near-duplicate paraphrases found in cluster ({len(matches)}):")
for m_id in sorted(matches)[:4]:
    idx = int(m_id.replace("row_", ""))
    print(f"  - [{idx:04d}]: {sample_subset[idx]}")

Seed Question: question about cancelling order {{Order Number}}
Near-duplicate paraphrases found in cluster (4):
  - [0000]: question about cancelling order {{Order Number}}
  - [0130]: have a question about cancelling order {{Order Number}}
  - [0053]: have a question about cancelling order {{Order Number}}
  - [0896]: question about cancelling order {{Order Number}}
